# 무릎 3D MRI로부터 pseudo X-ray 만들기

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhj0326/knee-pseudo-xray-demo/blob/main/knee_pseudo_xray.ipynb)

이 실습에서는 무릎 **3D MRI** 데이터로부터 **가짜 X-ray(pseudo X-ray)** 를 만드는 과정을
단계별로 확인하고, 같은 환자의 **실제 X-ray** 와 비교합니다.

### 학습 목표
- X-ray, MRI, pseudo X-ray가 각각 무엇인지 이해한다 (정의와 예시).
- X선이 몸을 통과하는 원리를 이해한다 (뼈는 많이, 살은 적게 막는다).
- 3D 데이터를 한 방향으로 더하는 **투영(projection)** 으로 2D pseudo X-ray를 만든다.
- 보는 각도에 따라 영상이 어떻게 달라지는지 관찰한다.
- 만든 pseudo X-ray와 실제 X-ray를 비교한다.

**핵심 개념.** X-ray 사진은 몸을 통과한 X선이 얼마나 약해졌는지를 기록한 영상입니다.
뼈처럼 단단한 곳은 X선을 많이 막아 밝게, 살처럼 무른 곳은 적게 막아 어둡게 보입니다.
이 실습에서는 3D 데이터를 한 방향으로 더해서 그 효과를 흉내 냅니다.

**실행 방법.** 상단 메뉴에서 "런타임 → 모두 실행"을 누르거나, 각 셀을 위에서부터 차례로 실행하세요.

참고: 그래프 안의 글자는 영어로 되어 있습니다(코랩에서 한글 폰트가 없어 깨지는 것을 막기 위함).
설명은 모두 한국어입니다.

## 핵심 용어 세 가지

본격적으로 만들기 전에, 실습에 나오는 세 영상을 간단히 정리합니다.
아래 "정의와 예시" 부분에서 실제 이미지로 하나씩 보여줍니다.

| 용어 | 정의 | 뼈가 보이는 방식 |
|---|---|---|
| X-ray | X선을 몸에 통과시켜 찍은 2차원 사진 | 뼈가 밝게(하얗게) |
| MRI (3D) | 자석과 전파로 몸속을 3차원으로 촬영한 영상 | 조직별로 구분 가능 |
| pseudo X-ray | 3D(MRI)를 가공해 만든 가짜 2차원 X-ray | 뼈가 밝게 (X-ray를 흉내) |

- X-ray는 싸고 빠르지만 2차원이라 앞뒤로 겹친 뼈가 서로 가려집니다.
- MRI는 3차원이라 정보가 풍부하지만, 비싸고 촬영 시간이 깁니다.
- pseudo X-ray는 MRI만 있으면 만들 수 있어, AI 학습 데이터를 늘리거나
  서로 다른 영상을 맞춰 비교하는 데 쓰입니다.

## 0. 데이터 내려받기

실습에 쓸 무릎 데이터(10명분)를 GitHub에서 내려받습니다. 환자마다
3D MRI 파일(`mri_label.npz`)과 실제 X-ray 사진(`real_xray.png`)이 들어 있습니다.

In [ ]:
import os

GITHUB_USER = "mhj0326"
REPO        = "knee-pseudo-xray-demo"

# data 폴더가 이미 있으면(로컬) 그대로 사용, 없으면 GitHub에서 clone
if os.path.exists("data/cases"):
    DATA = "data/cases"
elif os.path.exists(f"{REPO}/data/cases"):
    DATA = f"{REPO}/data/cases"
else:
    os.system(f"git clone https://github.com/{GITHUB_USER}/{REPO}.git")
    DATA = f"{REPO}/data/cases"

cases = sorted(os.listdir(DATA))
print("내려받은 환자 수:", len(cases), "명")
print("환자 목록:")
for i, c in enumerate(cases):
    print(f"  [{i}] {c}")

## 1. 도구 준비

계산(numpy), 그래프(matplotlib), 3D 회전(scipy), 이미지 읽기(PIL) 도구를 불러오고,
앞으로 계속 쓸 함수를 정의합니다.

- `load_case(이름)`: 한 환자의 3D MRI와 실제 X-ray를 불러온다.
- `make_pseudo_xray(볼륨, ...)`: 3D를 눌러서 pseudo X-ray를 만든다.

코랩에는 위 도구가 모두 기본 설치되어 있어 추가 설치가 필요 없습니다.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import rotate as nd_rotate
from PIL import Image

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

# 3D MRI 라벨 번호(0~7)의 의미.
#   한국어 이름(KO): 설명(print)용   /   영어 이름(EN): 그래프 안 글자용(한글 깨짐 방지)
CLASS_KO = ["배경(공기)", "뼈1 대퇴골", "뼈2 경골", "뼈3 슬개골",
            "연골/무른조직4", "연골/무른조직5", "연골/무른조직6", "연골/무른조직7"]
CLASS_EN = ["Background", "Bone1 Femur", "Bone2 Tibia", "Bone3 Patella",
            "Soft4", "Soft5", "Soft6", "Soft7"]

# 조직별 'X선을 막는 정도'(감쇠계수) 기본값: 뼈는 크게, 연골은 작게, 공기는 0
DEFAULT_MU = np.array([0.0, 1.0, 1.0, 1.0, 0.35, 0.35, 0.35, 0.35])

def load_case(name):
    """한 환자의 3D MRI 라벨 볼륨(128^3)과 실제 X-ray 이미지를 불러온다."""
    folder = os.path.join(DATA, name)
    vol  = np.load(os.path.join(folder, "mri_label.npz"))["vol"]   # (128,128,128), 값 0~7
    real = np.array(Image.open(os.path.join(folder, "real_xray.png")).convert("L"))
    return vol, real

def make_pseudo_xray(vol, mu=DEFAULT_MU, angle=0.0):
    """3D 볼륨을 angle도 회전한 뒤, 앞->뒤 방향으로 모두 더해(투영) pseudo X-ray를 만든다."""
    dens = mu[vol]                                   # 각 칸의 라벨 번호를 '막는 정도'로 변환
    if angle != 0:
        dens = nd_rotate(dens, angle, axes=(1, 2), reshape=False, order=1)
    pxray = dens.sum(axis=1)                          # 앞->뒤(axis=1)로 합산 = 투영
    return pxray                                       # 0행=위쪽. 대퇴골이 자연히 위로 온다

def normalize(img):
    """밝기를 0~1 범위로 맞춘다."""
    img = img.astype(np.float32)
    return (img - img.min()) / (img.max() - img.min() + 1e-8)

print("도구와 함수 준비 완료")

---
# 1부. 세 가지 영상의 정의와 예시

MRI → X-ray → pseudo X-ray 순서로, 각각이 무엇인지 설명하고 실제 예시 이미지를
하나씩 보여줍니다. 예시는 첫 번째 환자로 보여줍니다.

In [ ]:
EXAMPLE = cases[0]                 # 예시로 쓸 환자
ex_vol, ex_real = load_case(EXAMPLE)
print("예시 환자:", EXAMPLE)
print("3D MRI 크기:", ex_vol.shape, "= 128 x 128 x 128 (작은 정육면체 칸들)")

## 1-1. MRI (3D)

MRI(자기공명영상)는 강한 자석과 전파로 몸속을 **3차원**으로 촬영하는 장비입니다.
X선을 쓰지 않아 방사선이 없고, 근육이나 연골 같은 무른 조직까지 잘 보입니다.

이 실습의 MRI 데이터는 컴퓨터가 이미 조직마다 번호를 매겨 둔 3D 데이터입니다
(예: 대퇴골=1, 경골=2, 슬개골=3, 연골=4~7). 크기는 128 x 128 x 128칸입니다.

3D는 한 번에 볼 수 없으므로, 아래처럼 세 방향으로 잘라서(단면) 봅니다.
조직마다 다른 색으로 표시했습니다.

In [ ]:
mid = ex_vol.shape[0] // 2
views = [("Coronal (front)", ex_vol[:, mid, :]),
         ("Sagittal (side)", ex_vol[:, :, mid]),
         ("Axial (top)",     ex_vol[mid, :, :])]

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (title, sl) in zip(axes, views):
    ax.imshow(sl, cmap="nipy_spectral", vmin=0, vmax=7, interpolation="nearest")
    ax.set_title(title); ax.axis("off")
plt.suptitle(f"Example MRI (3D) slices  -  patient {EXAMPLE}   (color = tissue)", fontsize=13)
plt.tight_layout(); plt.show()

print("이 MRI에 들어 있는 조직:")
for k in np.unique(ex_vol):
    print(f"  번호 {k} = {CLASS_KO[int(k)]}")

### 3D로 회전시켜 보기

단면만 보면 입체 구조를 느끼기 어렵습니다. 이번에는 뼈 부분만 골라 3차원 점으로 그린 뒤,
조금씩 돌려 가며 여러 장면을 이어 붙여 **회전하는 GIF**를 만들어 봅니다.
무릎 뼈가 실제로 어떤 입체 모양인지 확인할 수 있습니다.

실행에 20~40초 정도 걸릴 수 있습니다.

In [ ]:
import io
from PIL import Image as PILImage

step  = 2                                      # 2칸마다 하나씩만 사용(가볍게)
small = ex_vol[::step, ::step, ::step]
zb, yb, xb = np.where((small >= 1) & (small <= 3))   # 뼈(번호 1~3)만 선택
cval = small[zb, yb, xb]                        # 뼈 종류(색 구분용)
ztop = small.shape[0] - zb                      # 대퇴골(낮은 z)이 위로 오도록

# 뼈별 색: 대퇴골/경골=뼈색(옅은 베이지), 슬개골=주황(눈에 띄게)
palette = {1: "#e6ddcf", 2: "#c9bfab", 3: "#e07a5f"}
pt_colors = np.array([palette[int(v)] for v in cval])

frames = []
for az in range(0, 360, 20):                    # 20도씩, 총 18장
    fig = plt.figure(figsize=(3.4, 4.4))
    ax = fig.add_subplot(111, projection="3d")
    ax.scatter(xb, yb, ztop, c=pt_colors, s=3, marker="s", linewidths=0)
    ax.view_init(elev=8, azim=az)
    ax.set_box_aspect((1, 1, 1.4))
    ax.set_axis_off()
    ax.set_xlim(0, small.shape[2]); ax.set_ylim(0, small.shape[1]); ax.set_zlim(0, small.shape[0])
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=70)
    plt.close(fig); buf.seek(0)
    frames.append(PILImage.open(buf).convert("RGB"))

gif_path = "knee_3d.gif"
frames[0].save(gif_path, save_all=True, append_images=frames[1:], duration=120, loop=0)
print("회전 GIF 저장 완료:", gif_path)

# 코랩/주피터에서 GIF를 바로 재생
try:
    from IPython.display import Image as IPYImage, display
    display(IPYImage(filename=gif_path))
except Exception:
    pass

## 1-2. X-ray

X-ray(엑스레이)는 몸 한쪽에서 X선을 쏘고, 반대쪽 센서에 도달한 X선의 양을 기록한
**2차원 사진**입니다.

- 뼈처럼 단단하고 밀도가 높은 곳은 X선을 많이 막아 센서에 적게 도달하고, 밝게 보입니다.
- 살이나 연골처럼 무른 곳은 X선을 적게 막아 센서에 많이 도달하고, 어둡게 보입니다.

장점은 싸고 빠르다는 것, 단점은 2차원이라 앞뒤로 겹친 뼈가 서로 가려진다는 점입니다.
아래는 같은 환자의 실제 X-ray(양쪽 다리 전체 사진)입니다. 가운데 세로 눈금은 길이 측정용 자입니다.

In [ ]:
plt.figure(figsize=(5, 6))
plt.imshow(ex_real, cmap="gray")
plt.title(f"Example real X-ray  -  patient {EXAMPLE}")
plt.axis("off"); plt.show()
print("밝은 부분은 뼈, 어두운 부분은 무른 조직이나 공기입니다.")

## 1-3. pseudo X-ray

pseudo X-ray는 실제로 X선을 쏘아 찍은 것이 아니라, 이미 가지고 있는 3D 데이터(MRI)를
컴퓨터로 가공해 X-ray처럼 보이게 만든 2차원 영상입니다. 'pseudo'는 '유사, 가짜'라는 뜻입니다.

만드는 원리는 다음과 같습니다. X선이 몸을 앞에서 뒤로 통과한다고 가정하고, 그 경로에 있는
조직들이 X선을 막는 정도를 모두 더합니다. 이를 **투영(projection)** 이라고 합니다.
많이 막힌 곳(뼈)을 밝게 표시하면 X-ray와 비슷한 영상이 됩니다.

$$\text{pseudo X-ray 픽셀값} \;=\; \sum_{\text{앞}\to\text{뒤}} (\text{그 경로 조직이 X선을 막는 정도})$$

실제 X-ray는 많이 모으기 어렵지만, pseudo X-ray는 MRI만 있으면 얼마든지 만들 수 있습니다.
그래서 AI 학습용 데이터를 늘리거나, MRI와 X-ray를 서로 맞춰 비교하는 데 유용합니다.

아래는 위 3D MRI로 만든 pseudo X-ray 예시입니다. 만드는 과정은 2부에서 단계별로 다룹니다.

In [ ]:
ex_pxray = normalize(make_pseudo_xray(ex_vol)) ** 0.6   # 0.6제곱: 대비를 키우는 보정

plt.figure(figsize=(4.5, 6))
plt.imshow(ex_pxray, cmap="gray")
plt.title(f"Example pseudo X-ray (from 3D MRI)  -  patient {EXAMPLE}")
plt.axis("off"); plt.show()
print("3D MRI를 앞->뒤로 더해서 만든 2차원 가짜 X-ray입니다.")

## 1-4. 세 가지 영상 비교

같은 환자의 MRI 단면, pseudo X-ray, 실제 X-ray를 한 줄로 놓고 비교합니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 5))
axes[0].imshow(ex_vol[:, mid, :], cmap="nipy_spectral", vmin=0, vmax=7,
               interpolation="nearest")
axes[0].set_title("MRI (3D slice)")
axes[1].imshow(ex_pxray, cmap="gray"); axes[1].set_title("Pseudo X-ray (made by us)")
axes[2].imshow(ex_real, cmap="gray");  axes[2].set_title("Real X-ray (hospital)")
for ax in axes: ax.axis("off")
plt.suptitle(f"Three image types  -  patient {EXAMPLE}", fontsize=13)
plt.tight_layout(); plt.show()

---
# 2부. pseudo X-ray를 단계별로 만들기

개념을 이해했으니, pseudo X-ray가 만들어지는 과정을 한 단계씩 살펴봅니다.

## 2-1. 실습할 환자 고르기

`PICK` 숫자(0~9)를 바꾸면 다른 환자로 실습할 수 있습니다.

In [ ]:
PICK = 0                      # 0 ~ 9 사이로 바꿔 보세요
CASE = cases[PICK]
vol, real = load_case(CASE)

print("선택한 환자:", CASE)
print("3D MRI 크기:", vol.shape)
print("들어 있는 조직 번호:", np.unique(vol))

## 2-2. 3D MRI 살펴보기

고른 환자의 3D MRI를 세 방향 단면으로 봅니다. 뼈(큰 덩어리)와 연골(가느다란 띠)의
위치를 눈으로 확인해 두세요. 이 데이터를 눌러서 X-ray로 만들 것입니다.

In [ ]:
mid = vol.shape[0] // 2
views = [("Coronal (front)", vol[:, mid, :]),
         ("Sagittal (side)", vol[:, :, mid]),
         ("Axial (top)",     vol[mid, :, :])]

fig, axes = plt.subplots(1, 3, figsize=(11, 4))
for ax, (title, sl) in zip(axes, views):
    ax.imshow(sl, cmap="nipy_spectral", vmin=0, vmax=7, interpolation="nearest")
    ax.set_title(title); ax.axis("off")
plt.suptitle(f"3D MRI slices  -  patient {CASE}", fontsize=13)
plt.tight_layout(); plt.show()

## 2-3. X선을 막는 정도 정하기

조직마다 X선을 막는 정도가 다릅니다. 단단한 뼈는 많이(값 크게), 무른 연골은 조금(값 작게),
공기는 전혀(0) 막습니다. 이 값을 `mu`(감쇠계수)라고 부릅니다.

In [ ]:
mu = np.array([0.0,   # 0 배경(공기)
               1.0,   # 1 뼈 (대퇴골)
               1.0,   # 2 뼈 (경골)
               1.0,   # 3 뼈 (슬개골)
               0.35,  # 4 연골/무른조직
               0.35,  # 5
               0.35,  # 6
               0.35]) # 7

plt.figure(figsize=(8, 3))
plt.bar(CLASS_EN, mu, color="steelblue")
plt.ylabel("X-ray attenuation (how much it blocks)")
plt.title("Attenuation per tissue  (bone = high, cartilage = low, air = 0)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()
print("뼈(1~3)는 1.0, 연골(4~7)은 0.35, 공기(0)는 0으로 정했습니다.")

## 2-4. 투영해서 pseudo X-ray 만들기 (가장 중요한 단계)

X선이 몸을 앞에서 뒤로 통과한다고 가정하고, 경로에 있는 모든 칸의 `mu` 값을 더합니다.
이것이 투영(projection)이며, 그 결과가 pseudo X-ray입니다.

- 더한 값이 큰 곳(뼈가 많은 경로)은 밝게 보입니다.
- 더한 값이 작은 곳(무른 조직만 있는 경로)은 어둡게 보입니다.

In [ ]:
pxray0 = make_pseudo_xray(vol, mu, angle=0)

fig, axes = plt.subplots(1, 2, figsize=(9, 5))
axes[0].imshow(vol[:, mid, :], cmap="nipy_spectral", vmin=0, vmax=7,
               interpolation="nearest")
axes[0].set_title("Input: 3D MRI (front slice)")
axes[1].imshow(pxray0, cmap="gray"); axes[1].set_title("Output: Pseudo X-ray (0 deg)")
for ax in axes: ax.axis("off")
plt.suptitle(f"Projection: 3D to 2D  -  patient {CASE}", fontsize=13)
plt.tight_layout(); plt.show()
print("왼쪽 3D를 앞->뒤로 모두 더해 오른쪽 2D pseudo X-ray를 만들었습니다.")

## 2-5. 여러 각도에서 바라보기

환자를 조금씩 돌려 가며 투영하면, 보는 각도에 따라 X-ray가 어떻게 달라지는지 알 수 있습니다.
실제 촬영에서도 다리가 살짝 돌아가면 사진이 달라집니다.

In [ ]:
angles = [-20, -10, 0, 10, 20]
fig, axes = plt.subplots(1, len(angles), figsize=(13, 5))
for ax, a in zip(axes, angles):
    ax.imshow(make_pseudo_xray(vol, mu, angle=a), cmap="gray")
    ax.set_title(f"{a} deg"); ax.axis("off")
plt.suptitle(f"Pseudo X-ray at different viewing angles  -  patient {CASE}", fontsize=13)
plt.tight_layout(); plt.show()

## 2-6. 실제 X-ray처럼 보정하기

투영한 그대로는 밋밋합니다. 밝기를 0~1로 맞추고(정규화), 감마 보정(값을 제곱)으로
뼈와 배경의 대비를 키우면 실제 X-ray에 더 가까워집니다. 감마가 1보다 작을수록
밝은 부분(뼈)이 더 또렷해집니다.

In [ ]:
raw   = normalize(pxray0)
gamma = raw ** 0.6

fig, axes = plt.subplots(1, 2, figsize=(8, 6))
axes[0].imshow(raw,   cmap="gray"); axes[0].set_title("Before (raw)")
axes[1].imshow(gamma, cmap="gray"); axes[1].set_title("After (gamma = 0.6)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 2-7. 실제 X-ray와 비교하기

만든 pseudo X-ray와 같은 환자의 실제 X-ray를 비교합니다. 완전히 같지는 않지만,
뼈의 모양과 관절 위치가 닮은 것을 확인할 수 있습니다.

참고: 실제 X-ray는 양쪽 다리 전체를 찍은 사진이라 가운데 자가 보이고,
pseudo X-ray는 한쪽 무릎의 3D 데이터만으로 만든 것입니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 6))
axes[0].imshow(gamma, cmap="gray"); axes[0].set_title("Pseudo X-ray (made from 3D MRI)")
axes[1].imshow(real,  cmap="gray"); axes[1].set_title("Real X-ray (hospital)")
for ax in axes: ax.axis("off")
plt.suptitle(f"Pseudo vs Real  -  patient {CASE}", fontsize=13)
plt.tight_layout(); plt.show()

## 2-8. 10명 전체 비교

10명 모두에 대해 3D MRI 단면, pseudo X-ray, 실제 X-ray를 한 줄씩 나열합니다.
사람마다 뼈 모양이 조금씩 다른 것을 관찰해 보세요.

In [ ]:
n = len(cases)
fig, axes = plt.subplots(n, 3, figsize=(8, 2.6 * n))
for i, name in enumerate(cases):
    v, r = load_case(name)
    m = v.shape[0] // 2
    axes[i, 0].imshow(v[:, m, :], cmap="nipy_spectral", vmin=0, vmax=7,
                      interpolation="nearest")
    axes[i, 1].imshow(normalize(make_pseudo_xray(v, mu)) ** 0.6, cmap="gray")
    axes[i, 2].imshow(r, cmap="gray")
    axes[i, 0].set_ylabel(name, fontsize=9)
    for j in range(3):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
axes[0, 0].set_title("MRI (3D slice)")
axes[0, 1].set_title("Pseudo X-ray")
axes[0, 2].set_title("Real X-ray")
plt.tight_layout(); plt.show()

## 직접 해보기

아래를 바꿔 가며 결과가 어떻게 달라지는지 실험해 보세요.

1. 2-1의 `PICK`을 0~9로 바꿔 다른 환자를 봅니다.
2. 아래 `mu_test`에서 연골 값(0.35)을 0.8처럼 키우면 어떻게 될까요? 연골도 뼈처럼 밝아질까요?
3. `make_pseudo_xray` 안에서 더하는 방향을 `axis=1`에서 `axis=2`로 바꾸면 옆에서 본 X-ray가 됩니다.
4. 감마 값(0.6)을 0.3이나 1.5로 바꿔 대비 변화를 관찰합니다.

In [ ]:
# 자유 실험: 연골도 X선을 많이 막는다면?
mu_test = np.array([0.0, 1.0, 1.0, 1.0, 0.8, 0.8, 0.8, 0.8])

fig, axes = plt.subplots(1, 2, figsize=(8, 6))
axes[0].imshow(normalize(make_pseudo_xray(vol, DEFAULT_MU)) ** 0.6, cmap="gray")
axes[0].set_title("Cartilage = 0.35 (default)")
axes[1].imshow(normalize(make_pseudo_xray(vol, mu_test)) ** 0.6, cmap="gray")
axes[1].set_title("Cartilage = 0.80 (changed)")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 정리

- MRI(3D)는 몸속을 3차원으로 담은 영상으로, 뼈와 연골 같은 조직을 구분할 수 있습니다.
- X-ray는 X선을 통과시켜 찍은 2차원 사진으로, 단단한 뼈가 밝게 보입니다.
- pseudo X-ray는 3D(MRI)를 한 방향으로 더해(투영) 만든 가짜 X-ray입니다.
- X선이 단단한 뼈를 많이, 무른 조직을 적게 막는다는 성질을 숫자(`mu`)로 흉내 냈습니다.
- 이렇게 만든 pseudo X-ray는 실제 X-ray와 뼈 모양 및 관절 위치가 닮아 있습니다.

이 원리는 의료 영상 AI에서 MRI로 가상의 X-ray를 만들어 데이터를 늘리거나,
서로 다른 영상을 비교하고 정렬하는 데 실제로 활용됩니다.